# ID Forensics — Colab Workbench

One notebook: setup → train → save weights → evaluate.

Everything persistent lives on **Google Drive** (`My Drive/id-forensics/`):
- `data/raw/` — images
- `outputs/` — model weights
- `eval/` — evaluation reports + wrong/correct image viz

**Colab Secrets** (🔑 left sidebar, allow notebook access):
- `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`
- `AWS_SESSION_TOKEN` *(only for temporary/SSO creds)*

**Run top to bottom.**

## Config

In [ ]:
GITHUB_TOKEN = ""   # only if repo is private

# True  = download new images from S3 → Drive (first run, or after new labels)
# False = images already on Drive, just rebuild splits (~30 sec)
SYNC_IMAGES = False

# Which stage to train: "stage1", "stage2", "stage4", or None to skip training
TRAIN_STAGE = "stage2"

# Run evaluation after training? Saves to Drive: id-forensics/eval/<stage>/<timestamp>/
RUN_EVAL = True
EVAL_SPLIT = "val"   # "val" or "test"

## 1. Setup

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO = Path("/content/id-forensics-model")
user, repo = "ansisvaisla", "id-forensics-model"
url = (
    f"https://{GITHUB_TOKEN}@github.com/{user}/{repo}.git"
    if GITHUB_TOKEN else f"https://github.com/{user}/{repo}.git"
)

os.chdir("/content")
if not REPO.is_dir():
    subprocess.run(["git", "clone", url, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "scripts"))
import colab_bootstrap as cb
importlib.reload(cb)

cb.setup(github_token=GITHUB_TOKEN, sync_images=SYNC_IMAGES)

## 2. Train Stage 2 — screen / printout / live

In [ ]:
STAGE2_EPOCHS = 40
STAGE2_BATCH = 32
DEVICE = "cuda"

In [ ]:
if TRAIN_STAGE == "stage2":
    !cd /content/id-forensics-model && python scripts/training/train_stage2_screen.py \
        --device {DEVICE} --epochs {STAGE2_EPOCHS} --batch {STAGE2_BATCH}
else:
    print("Skipping Stage 2 (TRAIN_STAGE != 'stage2')")

## 3. Save weights to Drive

In [ ]:
if TRAIN_STAGE == "stage2":
    cb.save_weights("stage2")
elif TRAIN_STAGE == "stage1":
    cb.save_weights("stage1")
elif TRAIN_STAGE == "stage4":
    cb.save_weights("stage4")
else:
    print("No weights to save (TRAIN_STAGE is None)")

## 4. Evaluate (saved to Drive)

Outputs persist at `My Drive/id-forensics/eval/<stage>/<timestamp>/`:
- `report.txt` — accuracy, per-class metrics
- `viz/wrong/` — misclassified images to review
- `confusion_matrix.png`, `predictions.csv`

In [ ]:
if RUN_EVAL and TRAIN_STAGE:
    eval_dir = cb.run_eval(TRAIN_STAGE, split=EVAL_SPLIT)
    print(f"\nBrowse wrong predictions in Drive:\n  {eval_dir / 'viz' / 'wrong'}")
else:
    print("Skipping eval (RUN_EVAL=False or TRAIN_STAGE=None)")

---
## Optional: Stage 1 (corner detection)

Set `TRAIN_STAGE = "stage1"` in Config, then uncomment and run:

In [ ]:
# cb.clear_corner_caches()
# !cd /content/id-forensics-model && python scripts/training/train_stage1_corners.py \
#     --model yolov8s-pose.pt --device 0 --epochs 100 --batch 16

## Optional: Stage 4 (ID type classifier)

Set `TRAIN_STAGE = "stage4"` in Config. Requires Stage 1 weights for deskewing.

In [ ]:
# !cd /content/id-forensics-model && python scripts/deskew_id_type_images.py
# !cd /content/id-forensics-model && python scripts/split_id_type_dataset.py --source all_deskewed
# !cd /content/id-forensics-model && python scripts/training/train_stage4_id_type.py \
#     --device cuda --epochs 40 --batch 32